# Setup and loading models from /models folder

In [1]:
import pickle
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent
MODEL_DIR = PROJECT_ROOT / "models"

model_names = ["RW", "DNS", "Ridge", "XGBoost"]

model_preds = {}
model_actuals = {}
model_metrics = {}

for name in model_names:
    path = MODEL_DIR / f"{name}.pkl"
    with open(path, "rb") as f:
        bundle = pickle.load(f)

    print(f"Loaded {name} from {path}, keys: {list(bundle.keys())}")

    model_preds[name]   = bundle["predictions"]
    model_actuals[name] = bundle["actuals"]
    model_metrics[name] = bundle["metrics"]   # <--- NEW

print("Models loaded:", list(model_preds.keys()))


Loaded RW from /Users/chriss/Desktop/Studium/Semester/7. Semester/Bachelorarbeit/yield-curve-forecasting/yield-curve-forecasting/models/RW.pkl, keys: ['predictions', 'actuals', 'metrics']
Loaded DNS from /Users/chriss/Desktop/Studium/Semester/7. Semester/Bachelorarbeit/yield-curve-forecasting/yield-curve-forecasting/models/DNS.pkl, keys: ['predictions', 'actuals', 'metrics']
Loaded Ridge from /Users/chriss/Desktop/Studium/Semester/7. Semester/Bachelorarbeit/yield-curve-forecasting/yield-curve-forecasting/models/Ridge.pkl, keys: ['predictions', 'actuals', 'metrics', 'hyperparameters']
Loaded XGBoost from /Users/chriss/Desktop/Studium/Semester/7. Semester/Bachelorarbeit/yield-curve-forecasting/yield-curve-forecasting/models/XGBoost.pkl, keys: ['predictions', 'actuals', 'metrics', 'hyperparameters']
Models loaded: ['RW', 'DNS', 'Ridge', 'XGBoost']


In [5]:
import pandas as pd
import numpy as np
from pathlib import Path

# ----------------- Paths -----------------
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"

# ----------------- Load raw FRED data -----------------
DGS1 = pd.read_csv(DATA_DIR / "DGS1.csv")
DGS2 = pd.read_csv(DATA_DIR / "DGS2.csv")
DGS5 = pd.read_csv(DATA_DIR / "DGS5.csv")
DGS10 = pd.read_csv(DATA_DIR / "DGS10.csv")

# 2y, 5y, 10y panel
merged = (
    DGS2.merge(DGS5, on="observation_date", how="inner")
        .merge(DGS10, on="observation_date", how="inner")
)
merged = merged.dropna(subset=["DGS2", "DGS5", "DGS10"])
merged = merged.set_index("observation_date")
merged.index = pd.to_datetime(merged.index, errors="coerce")

# Short rate (1y) as separate series
DGS1 = DGS1.dropna(subset=["DGS1"])
DGS1 = DGS1.set_index("observation_date")
DGS1.index = pd.to_datetime(DGS1.index, errors="coerce")

short_rate = DGS1["DGS1"]   # <-- THIS is what you pass to econ functions

# ----------------- Settings -----------------
maturity_cols = ["DGS2", "DGS5", "DGS10"]
horizons = [1, 5, 10, 30]
idx = merged.index
train_start = pd.Timestamp("2009-01-02")
train_end   = pd.Timestamp("2018-12-31")

test_start  = pd.Timestamp("2019-01-02")
test_end    = pd.Timestamp("2025-11-25")  

# Duration proxy (years-to-maturity)
maturity_durations = {
    "DGS2": 2.0,
    "DGS5": 5.0,
    "DGS10": 10.0
}



## Long–Short Trading Strategy

To assess whether yield forecasts translate into economic value, we implement a simple forecast-based long–short trading strategy following Carriero et al. (2012). The strategy is evaluated separately for each maturity (2-year, 5-year, and 10-year) and each forecast horizon  
\( h \in \{1,5,10,30\} \) trading days.

At each forecast date \(t\), the model produces a forecast of the excess holding-period return \( \widehat{er}_{t+h} \). A directional trading position is formed based solely on the sign of this forecast:

- \( \widehat{er}_{t+h} > 0 \): long position  
- \( \widehat{er}_{t+h} < 0 \): short position  
- \( \widehat{er}_{t+h} = 0 \): no position  

Formally, the trading signal is
\[
s_{t+h} = \operatorname{sign}(\widehat{er}_{t+h}).
\]

The realized excess holding-period return \( er_{t+h} \) is computed over the same horizon. Trading profits are then given by
\[
\pi_{t+h} = s_{t+h} \cdot er_{t+h} \cdot 100,
\]
where the constant factor represents a fixed notional investment and does not affect relative model comparisons.

Performance is evaluated over the out-of-sample period using the time series of trading profits. We report average trading profit, the standard deviation of profits, cumulative profit, and the Sharpe ratio, defined as the mean trading profit divided by its population standard deviation (not annualized). Sharpe ratios are interpreted as relative risk-adjusted performance measures and are comparable across models under the same trading rule.


In [6]:
def compute_predicted_excess_return(pred_yield_series, merged_yields, short_rate,
                                    maturity_col, horizon, trading_days=252):
    D = maturity_durations[maturity_col]
    idx = pred_yield_series.index  # t+h

    y_start = merged_yields[maturity_col].shift(horizon).loc[idx]
    y_end_pred = pred_yield_series

    # convert yield change from percentage points to decimal
    dy_pred = (y_end_pred - y_start) / 100.0
    r_hat = -D * dy_pred

    # convert short rate (percent p.a.) to horizon return in decimal
    short_at_end = (short_rate.loc[idx] / 100.0) * (horizon / trading_days)

    er_pred = r_hat - short_at_end
    return er_pred.dropna()


def compute_realized_excess_return(merged_yields, short_rate, maturity_col,
                                   horizon, idx, trading_days=252):
    D = maturity_durations[maturity_col]
    y = merged_yields[maturity_col]

    y_end = y.loc[idx]
    y_start = y.shift(horizon).loc[idx]

    dy_real = (y_end - y_start) / 100.0
    r_real = -D * dy_real

    short_at_end = (short_rate.loc[idx] / 100.0) * (horizon / trading_days)

    er_real = r_real - short_at_end
    return er_real.dropna()

def sharpe_ratio(pi_series):
    """
    Sharpe = mean(π) / std(π), mit Populations-Std
    """
    mu = pi_series.mean()
    sigma = pi_series.std(ddof=0)
    return mu / sigma if sigma != 0 else np.nan



In [10]:
# =====================================================
# Trading strategy: forecast-based long/short
# =====================================================

short_rate = DGS1["DGS1"]  

idx = merged.index

model_names_trading = ["RW", "DNS", "Ridge", "XGBoost"]
model_pred_series = {name: model_preds[name] for name in model_names_trading}

# any model's actuals ok; using RW just as canonical key set
actual_series_dict = model_actuals["RW"]

trading_results = []

for maturity in maturity_cols:
    for h in horizons:
        key = (maturity, h)

        if key not in actual_series_dict:
            continue

        # Realized excess returns (same for all models)
        er_realized = compute_realized_excess_return(
            merged_yields=merged,
            short_rate=short_rate,
            maturity_col=maturity,
            idx=idx,
            horizon=h,
        )

        for model_name, pred_dict in model_pred_series.items():
            if key not in pred_dict:
                continue

            pred_yields = pred_dict[key]

            # Predicted excess returns for this model/maturity/horizon
            er_pred = compute_predicted_excess_return(
                pred_yield_series=pred_yields,
                merged_yields=merged,
                short_rate=short_rate,
                maturity_col=maturity,
                horizon=h,
            )

            # Align realized and predicted excess returns
            df_tmp = (
                pd.DataFrame({"er_realized": er_realized})
                .join(er_pred.rename("er_pred"), how="inner")
                .dropna()
            )
            if df_tmp.empty:
                continue

            # -----------------------------------------------------
            # Trading P&L (Carriero TR1):
            # π_{t+h} = sign(er̂_{t+h}) * er_{t+h} * 100
            # -----------------------------------------------------
            pnl = np.sign(df_tmp["er_pred"]) * df_tmp["er_realized"] * 100.0

            avg_profit = pnl.mean()
            sr = sharpe_ratio(pnl)
            N = len(pnl)

            # -----------------------------------------------------
            # Hit ratio based on yield-change direction:
            # sign(Δy_pred) == sign(Δy_real)
            # -----------------------------------------------------
            eval_idx = df_tmp.index  # indexed by t+h

            y_start = merged[maturity].shift(h).loc[eval_idx]   # y_t
            y_end_real = merged[maturity].loc[eval_idx]         # y_{t+h}
            dy_real = (y_end_real - y_start)

            y_end_pred = pred_yields.loc[eval_idx]              # ŷ_{t+h}
            dy_pred = (y_end_pred - y_start)


            trading_results.append({
                "Maturity":   maturity,
                "Horizon":    h,
                "Model":      model_name,
                "Avg_Profit": avg_profit,
                "Sharpe":     sr,
                "N_Trades":   N,
            })

trading_results_df = (
    pd.DataFrame(trading_results)
    .sort_values(["Horizon", "Maturity", "Model"])
)

display(trading_results_df)


,Maturity,Horizon,Model,Avg_Profit,Sharpe,N_Trades
33,DGS10,1,DNS,0.005325,0.009020,1725
32,DGS10,1,RW,0.018361,0.031123,1726
34,DGS10,1,Ridge,0.002795,0.004733,1725
35,DGS10,1,XGBoost,0.001597,0.002697,1712
1,DGS2,1,DNS,0.011792,0.096048,1725
0,DGS2,1,RW,0.011814,0.096255,1726
2,DGS2,1,Ridge,0.011034,0.089826,1725
3,DGS2,1,XGBoost,0.004860,0.039319,1712
17,DGS5,1,DNS,0.015444,0.049344,1725
16,DGS5,1,RW,0.013726,0.043854,1726


In [36]:
trading_results_df[trading_results_df["Horizon"] == 30]

,Maturity,Horizon,Model,Avg_Profit,Sharpe,Hit_Ratio,N_Trades
33,DGS10,30,DNS,0.379732,0.114297,0.460145,1696
34,DGS10,30,Ridge,0.062228,0.018612,0.463164,1696
35,DGS10,30,XGBoost,-0.054254,-0.016165,0.474741,1683
9,DGS2,30,DNS,0.356992,0.466077,0.512030,1696
10,DGS2,30,Ridge,0.357056,0.466179,0.458976,1696
11,DGS2,30,XGBoost,0.274784,0.342678,0.558385,1683
21,DGS5,30,DNS,0.419740,0.237266,0.411343,1696
22,DGS5,30,Ridge,0.376238,0.211509,0.416119,1696
23,DGS5,30,XGBoost,0.122118,0.067067,0.493381,1683


In [11]:
best_by_sharpe = (
    trading_results_df
    .loc[trading_results_df.groupby(["Maturity", "Horizon"])["Sharpe"].idxmax()]
    .sort_values(["Horizon", "Maturity"])
)

display(best_by_sharpe)


,Maturity,Horizon,Model,Avg_Profit,Sharpe,N_Trades
32,DGS10,1,RW,0.018361,0.031123,1726
0,DGS2,1,RW,0.011814,0.096255,1726
17,DGS5,1,DNS,0.015444,0.049344,1725
36,DGS10,5,RW,0.091864,0.072399,1726
6,DGS2,5,Ridge,0.060162,0.226424,1721
21,DGS5,5,DNS,0.075092,0.111831,1721
40,DGS10,10,RW,0.183380,0.102265,1726
9,DGS2,10,DNS,0.119098,0.307745,1716
25,DGS5,10,DNS,0.146670,0.153814,1716
44,DGS10,30,RW,0.521694,0.158731,1726


# Risk Reduction Strategy
To assess whether yield forecasts provide practical economic value, we implement a timing strategy aimed at reducing interest-rate risk. The investor holds an equal-weighted portfolio of zero-coupon U.S. Treasury bonds (2y, 5y, 10y) and we compare two approaches:

Buy-and-Hold (BH):
The investor remains fully invested throughout the entire out-of-sample period.

Forecast-Based Timing Strategy:
At each forecast horizon 
ℎ
∈
{
1
,
5
,
10
,
30
}
h∈{1,5,10,30}, the investment position depends on the sign of the forecasted excess return:

Position rule (plain text version):

If forecasted excess return > 0 → stay invested (position = 1)
If forecasted excess return ≤ 0 → move to cash (position = 0)


No leverage or short-selling is allowed.

Returns are measured as excess returns relative to the short-term rate, so holding cash corresponds to earning zero excess return.

Performance is evaluated using:

mean excess return

volatility

Sharpe ratio

maximum drawdown

A forecasting model is considered economically useful if it reduces drawdowns or improves risk-adjusted performance relative to Buy-and-Hold — especially in the rising-rate environment during 2019–2025.

In [9]:
import numpy as np
import pandas as pd

TRADING_DAYS_PER_YEAR = 252

# Core configuration (only defined once in the notebook)
maturity_cols = ["DGS2", "DGS5", "DGS10"]
tau_years = np.array([2.0, 5.0, 10.0])  # in years


def compute_horizon_bond_returns(yields_dec, tau_years, h):
    """
    Approximate h-period holding returns for each maturity using
        r_{t,t+h}(τ) ≈ -τ (y_{t+h} - y_t),
    where yields are in decimal (not percent).

    Parameters
    ----------
    yields_dec : DataFrame
        DataFrame of yields in decimal form (e.g. 0.02, 0.05, ...)
        index = dates, columns = maturities (e.g. ['DGS2','DGS5','DGS10']).
    tau_years : array-like
        Vector of maturities in years, aligned with columns of yields_dec.
    h : int
        Horizon in trading days.

    Returns
    -------
    DataFrame
        Index = t (start dates), columns = maturities.
    """
    cols = list(yields_dec.columns)
    idx = yields_dec.index.to_list()

    rows = []
    dates = []

    for i in range(len(idx) - h):
        t   = idx[i]
        t_h = idx[i + h]

        y_t  = yields_dec.loc[t].values
        y_th = yields_dec.loc[t_h].values

        r_vec = -tau_years * (y_th - y_t)
        rows.append(r_vec)
        dates.append(t)

    r_df = pd.DataFrame(rows, index=pd.DatetimeIndex(dates), columns=cols)
    return r_df


def max_drawdown(return_series):
    """
    Computes max drawdown of the cumulative wealth path implied by returns.

    Parameters
    ----------
    return_series : Series
        Series of (excess) returns per period.

    Returns
    -------
    float
        Minimum drawdown (negative number).
    """
    cum = (1 + return_series).cumprod()
    peak = cum.cummax()
    dd = cum / peak - 1.0
    return dd.min()


In [14]:
def _strategy_metrics(excess_series):
    """
    Compute mean, std, Sharpe, max drawdown, cumulative return,
    and final wealth for a given excess-return series.
    """
    mean = excess_series.mean()
    std = excess_series.std(ddof=1)
    sharpe = mean / std if std > 0 else np.nan
    mdd = max_drawdown(excess_series)
    cumret = (1 + excess_series).prod() - 1
    final_wealth = (1 + excess_series).cumprod().iloc[-1]
    return mean, std, sharpe, mdd, cumret, final_wealth


def run_portfolio_timing_overlay(
    model_name,
    pred_dict,
    merged,
    short_rate,
    maturity_cols=maturity_cols,
    tau_years=tau_years,
    horizons=None,
    weights=None,
    test_start=pd.Timestamp("2019-01-02"),
    test_end=pd.Timestamp("2025-11-25"),
):
    """
    Portfolio-timing overlay:
    - Base portfolio: long in all maturities with given weights.
    - Benchmark (BH): always invested (buy & hold).
    - Timing overlay: invest only when forecasted portfolio excess return > 0,
      otherwise stay in cash (no bond exposure, no leverage).

    Parameters
    ----------
    model_name : str
        Name of the forecasting model (for labeling).
    pred_dict : dict
        Dictionary {(maturity, horizon): forecast_yield_series}.
        This should come from model_preds[model_name].
    merged : DataFrame
        Yield panel with columns containing maturity_cols in percent (e.g. 2.0, 3.5).
    short_rate : Series
        Short rate (e.g. DGS1) in percent, indexed by date.
    maturity_cols : list
        List of maturity column names in merged.
    tau_years : array-like
        Vector of maturities in years, aligned with maturity_cols.
    horizons : list or None
        List of horizons in trading days. If None, defaults to [1, 5, 10, 30].
    weights : list or None
        Portfolio weights across maturities. If None, equal weights.
    test_start, test_end : Timestamp
        Out-of-sample evaluation window.

    Returns
    -------
    summary_df : DataFrame
        Summary metrics per horizon.
    detail_by_h : dict
        detail_by_h[h] -> DataFrame with columns ["BH_excess", "Timing_excess"].
    """
    if horizons is None:
        horizons = [1, 5, 10, 30]

    # Normalize portfolio weights
    if weights is None:
        weights = np.ones(len(maturity_cols)) / len(maturity_cols)
    else:
        weights = np.array(weights, dtype=float)
        weights = weights / weights.sum()

    results = []
    detail_by_h = {}

    # Yields in decimal once (DRY)
    yields_dec = merged[maturity_cols] / 100.0

    for h in horizons:
        print(f"\n=== {model_name} | horizon h={h} ===")

        # 1) Realized bond returns r_{t,t+h} for each maturity (decimal)
        r_all = compute_horizon_bond_returns(yields_dec, tau_years, h)
        # restrict to test window
        r_all = r_all.loc[(r_all.index >= test_start) & (r_all.index <= test_end)]

        # 2) Find common dates where we have forecasts for all maturities
        date_lists = []
        for col in maturity_cols:
            key = (col, h)
            if key not in pred_dict:
                date_lists = []
                break
            date_lists.append(pred_dict[key].index)

        if not date_lists:
            print(f"  -> No forecasts for horizon {h}, skipping.")
            continue

        common_dates = sorted(set(date_lists[0]).intersection(*date_lists[1:]))
        common_dates = [d for d in common_dates if (d >= test_start) and (d <= test_end)]

        # Intersection with dates where we have realized returns
        common_dates = sorted(set(common_dates).intersection(r_all.index))
        if not common_dates:
            print("  -> No common dates for forecasts and returns.")
            continue

        # Option: rebalance every h days (non-overlapping windows)
        rebalance_dates = common_dates[::h]

        bh_excess = []
        timing_excess = []

        for t in rebalance_dates:
            if t not in r_all.index:
                continue

            # realized bond return vector for t->t+h
            r_vec = r_all.loc[t].values  # shape (n_maturities,)
            r_port = np.dot(weights, r_vec)

            # risk-free h-period return from short_rate (DGS1)
            if t in short_rate.index:
                rf_t = short_rate.loc[t] / 100.0  # decimal
                rf_h = rf_t * (h / TRADING_DAYS_PER_YEAR)
            else:
                rf_h = 0.0

            # realized excess portfolio return
            er_port = r_port - rf_h

            # forecasts: y_t and y_hat_{t+h|t}
            if t not in yields_dec.index:
                continue
            y_t = yields_dec.loc[t].values  # actual yields at t (decimal)

            try:
                y_hat_vals = np.array(
                    [pred_dict[(col, h)].loc[t] for col in maturity_cols]
                ) / 100.0  # to decimal
            except KeyError:
                # missing forecast at t for some maturity
                continue

            # predicted bond returns per maturity
            r_hat_vec = -tau_years * (y_hat_vals - y_t)
            r_hat_port = np.dot(weights, r_hat_vec)
            er_hat_port = r_hat_port - rf_h  # expected excess portfolio return

            # Benchmark: always invested
            bh_excess.append((t, er_port))

            # Timing: invest only if expected excess return > 0
            position = 1.0 if er_hat_port > 0 else 0.0
            timing_excess.append((t, position * er_port))

        # 3) Build DataFrames
        if not bh_excess:
            print("  -> No valid observations, skipping.")
            continue

        bh_df = (
            pd.DataFrame(bh_excess, columns=["date", "BH_excess"])
            .set_index("date")
        )
        timing_df = (
            pd.DataFrame(timing_excess, columns=["date", "Timing_excess"])
            .set_index("date")
        )

        # align indexes
        idx_common = bh_df.index.intersection(timing_df.index)
        bh_df = bh_df.loc[idx_common]
        timing_df = timing_df.loc[idx_common]

        detail_by_h[h] = pd.concat([bh_df, timing_df], axis=1)

        # 4) Metrics (using helper to keep DRY)
        bh_series = detail_by_h[h]["BH_excess"]
        tm_series = detail_by_h[h]["Timing_excess"]
        N = len(bh_series)

        bh_mean, bh_std, bh_sharpe, bh_mdd, bh_cumret, bh_final_wealth = _strategy_metrics(bh_series)
        tm_mean, tm_std, tm_sharpe, tm_mdd, tm_cumret, tm_final_wealth = _strategy_metrics(tm_series)

        risk_reduction_std = (
            1.0 - (tm_std / bh_std) if bh_std > 0 and tm_std >= 0 else np.nan
        )

        print(f"  N={N}")
        print(
            f"  Buy&Hold    : mean={bh_mean:.6f}, std={bh_std:.6f}, "
            f"Sharpe={bh_sharpe:.3f}, MDD={bh_mdd:.3%}"
        )
        print(
            f"  Timing({model_name}): mean={tm_mean:.6f}, std={tm_std:.6f}, "
            f"Sharpe={tm_sharpe:.3f}, MDD={tm_mdd:.3%}"
        )
        print(f"  Std risk reduction: {risk_reduction_std:.3%}")

        results.append({
            "Model": model_name,
            "Horizon": h,
            "N": N,
            "BH_Mean": bh_mean,
            "BH_Std": bh_std,
            "BH_Sharpe": bh_sharpe,
            "BH_MaxDD": bh_mdd,
            "BH_CumRet": bh_cumret,
            "BH_FinalWealth": bh_final_wealth,
            "Timing_Mean": tm_mean,
            "Timing_Std": tm_std,
            "Timing_Sharpe": tm_sharpe,
            "Timing_MaxDD": tm_mdd,
            "Timing_CumRet": tm_cumret,
            "Timing_FinalWealth": tm_final_wealth,
            "Std_Risk_Reduction": risk_reduction_std,
        })

    summary_df = pd.DataFrame(results)
    return summary_df, detail_by_h



In [18]:
# Use the loaded model prediction dictionaries from the .pkl files
predictions = {
    "DNS":     model_preds["DNS"],
    "Ridge":   model_preds["Ridge"],
    "XGBoost": model_preds["XGBoost"],
    # optionally include RW if desired:
    # "RW": model_preds["RW"],
}

overlay_results = []
overlay_details = {}

for model_name, pred_dict in predictions.items():
    print(f"\n>>> Running timing overlay for {model_name}...")
    
    summary, details = run_portfolio_timing_overlay(
        model_name=model_name,
        pred_dict=pred_dict,
        merged=merged,
        short_rate=short_rate,
        horizons=[1,5,10,30],
        test_start=test_start,
        test_end=test_end
    )

    overlay_results.append(summary.assign(Model=model_name))
    overlay_details[model_name] = details

overlay_results_df = pd.concat(overlay_results, ignore_index=True)

overlay_results_df



>>> Running timing overlay for DNS...

=== DNS | horizon h=1 ===
  N=1724
  Buy&Hold    : mean=-0.000151, std=0.003310, Sharpe=-0.045, MDD=-31.536%
  Timing(DNS): mean=-0.000038, std=0.002276, Sharpe=-0.017, MDD=-13.751%
  Std risk reduction: 31.242%

=== DNS | horizon h=5 ===
  N=344
  Buy&Hold    : mean=-0.000723, std=0.007238, Sharpe=-0.100, MDD=-31.208%
  Timing(DNS): mean=-0.000705, std=0.004872, Sharpe=-0.145, MDD=-23.265%
  Std risk reduction: 32.693%

=== DNS | horizon h=10 ===
  N=171
  Buy&Hold    : mean=-0.001490, std=0.010018, Sharpe=-0.149, MDD=-31.058%
  Timing(DNS): mean=-0.000387, std=0.007581, Sharpe=-0.051, MDD=-18.085%
  Std risk reduction: 24.323%

=== DNS | horizon h=30 ===
  N=56
  Buy&Hold    : mean=-0.004559, std=0.018112, Sharpe=-0.252, MDD=-30.522%
  Timing(DNS): mean=-0.003396, std=0.012472, Sharpe=-0.272, MDD=-20.001%
  Std risk reduction: 31.140%

>>> Running timing overlay for Ridge...

=== Ridge | horizon h=1 ===
  N=1724
  Buy&Hold    : mean=-0.000151, 

,Model,Horizon,N,BH_Mean,BH_Std,BH_Sharpe,BH_MaxDD,BH_CumRet,BH_FinalWealth,Timing_Mean,Timing_Std,Timing_Sharpe,Timing_MaxDD,Timing_CumRet,Timing_FinalWealth,Std_Risk_Reduction
0,DNS,1,1724,-0.000151,0.003310,-0.045500,-0.315356,-0.235901,0.764099,-0.000038,0.002276,-0.016861,-0.137509,-0.068172,0.931828,0.312420
1,DNS,5,344,-0.000723,0.007238,-0.099823,-0.312083,-0.227109,0.772891,-0.000705,0.004872,-0.144698,-0.232647,-0.218590,0.781410,0.326934
2,DNS,10,171,-0.001490,0.010018,-0.148779,-0.310577,-0.231717,0.768283,-0.000387,0.007581,-0.051051,-0.180852,-0.068609,0.931391,0.243231
3,DNS,30,56,-0.004559,0.018112,-0.251709,-0.305222,-0.232792,0.767208,-0.003396,0.012472,-0.272314,-0.200010,-0.177042,0.822958,0.311403
4,Ridge,1,1724,-0.000151,0.003310,-0.045500,-0.315356,-0.235901,0.764099,-0.000033,0.002272,-0.014597,-0.125097,-0.059761,0.940239,0.313507
5,Ridge,5,344,-0.000723,0.007238,-0.099823,-0.312083,-0.227109,0.772891,-0.000640,0.004759,-0.134508,-0.209813,-0.200816,0.799184,0.342498
6,Ridge,10,171,-0.001490,0.010018,-0.148779,-0.310577,-0.231717,0.768283,-0.000417,0.007609,-0.054849,-0.190015,-0.073465,0.926535,0.240407
7,Ridge,30,56,-0.004559,0.018112,-0.251709,-0.305222,-0.232792,0.767208,-0.003336,0.012510,-0.266631,-0.200010,-0.174260,0.825740,0.309262
8,XGBoost,1,1712,-0.000154,0.003318,-0.046553,-0.315356,-0.239592,0.760408,-0.000063,0.002268,-0.027657,-0.173454,-0.105770,0.894230,0.316495
9,XGBoost,5,342,-0.000745,0.007249,-0.102832,-0.312083,-0.232007,0.767993,-0.000677,0.005247,-0.128950,-0.232108,-0.210359,0.789641,0.276120


In [22]:
# 1. Compute per-row improvements timing vs buy-and-hold
tmp = (
    overlay_results_df
    .assign(
        Mean_Delta      = lambda d: d["Timing_Mean"]   - d["BH_Mean"],
        Sharpe_Delta    = lambda d: d["Timing_Sharpe"] - d["BH_Sharpe"],
        Vol_Reduction   = lambda d: 1 - d["Timing_Std"] / d["BH_Std"],
        MaxDD_Reduction = lambda d: 1 - d["Timing_MaxDD"].abs() / d["BH_MaxDD"].abs()
    )
)

# 2. Aggregate across horizons → one row per model
summary_table = (
    tmp
    .groupby("Model")
    .agg(
        Mean_ER_Delta       = ("Mean_Delta", "mean"),   # avg per-step excess return improvement
        Sharpe_Delta        = ("Sharpe_Delta", "mean"), # avg Sharpe improvement
        Vol_Reduction_pct   = ("Vol_Reduction",   lambda x: 100 * x.mean()),
        MaxDD_Reduction_pct = ("MaxDD_Reduction", lambda x: 100 * x.mean()),
    )
    .reset_index()
)

# 3. Nice rounding for thesis tables
summary_table_rounded = summary_table.copy()
summary_table_rounded["Mean_ER_Delta"]       = summary_table_rounded["Mean_ER_Delta"].round(5)
summary_table_rounded["Sharpe_Delta"]        = summary_table_rounded["Sharpe_Delta"].round(3)
summary_table_rounded["Vol_Reduction_pct"]   = summary_table_rounded["Vol_Reduction_pct"].round(1)
summary_table_rounded["MaxDD_Reduction_pct"] = summary_table_rounded["MaxDD_Reduction_pct"].round(1)

summary_table_rounded


,Model,Mean_ER_Delta,Sharpe_Delta,Vol_Reduction_pct,MaxDD_Reduction_pct
0,DNS,0.00060,0.015,29.8,39.5
1,Ridge,0.00062,0.019,30.1,41.6
2,XGBoost,0.00044,-0.003,27.4,30.7
